In [1]:
import pandas as pd
import plotly.express as px
from ipywidgets import interact, widgets
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import plotly.express as px


In [2]:
# CSV-Dateien laden
df1 = pd.read_csv("final_df_mimic.csv")
df2 = pd.read_csv("final_df_smart_company.csv")
df3 = pd.read_csv("final_df_kropt.csv")

def get_df(name):
    return {'df1': df1, 'df2': df2, 'df3': df3}[name]

In [3]:


custom_colors = px.colors.qualitative.Set3

@interact(df_name=['df1', 'df2', 'df3'])
def plot_histogram(df_name):
    df = get_df(df_name)
    numeric_cols = df.select_dtypes(include='number').columns

    @interact(feature=numeric_cols)
    def inner_plot(feature):
        fig = px.histogram(
            df,
            x=feature,
            nbins=10,
            color_discrete_sequence=custom_colors,
            title=f"Verteilung von {feature}"
        )

        # Werte fett über Balken anzeigen
        fig.update_traces(
            texttemplate="<b>%{y}</b>",
            textposition="outside",
            marker_line_color="black",   # schwarze Konturen für bessere Trennung
            marker_line_width=1.2
        )

        fig.update_layout(
            title_x=0.5,
            bargap=0.15
        )

        fig.show()


interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …

In [ ]:
custom_colors = ["#5B5B5B", "#D24740", "#1F4E79", "#E4B600", "#000000", "#BFA74A"]
@interact(df_name=['df1', 'df2', 'df3'])
def show_pairplot(df_name):
    df = get_df(df_name)
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) >= 2:
        sns.pairplot(df[numeric_cols[:5]])


interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …

In [5]:
custom_colors = ["#5B5B5B", "#D24740", "#1F4E79", "#E4B600", "#000000", "#BFA74A"]
@interact(df_name=['df1', 'df2', 'df3'])
def plot_scatter(df_name):
    df = get_df(df_name)
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) >= 2:
        @interact(x=numeric_cols, y=numeric_cols)
        def inner_scatter(x, y):
            if x != y:
                fig = px.scatter(df, x=x, y=y, title=f"{y} vs {x}")
                fig.show()

interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …

In [ ]:
custom_colors = ["#5B5B5B", "#D24740", "#1F4E79", "#E4B600", "#000000", "#BFA74A"]
@interact(df_name=['df1', 'df2', 'df3'])
def plot_correlation(df_name):
    df = get_df(df_name)
    numeric_df = df.select_dtypes(include='number')
    if not numeric_df.empty:
        fig = px.imshow(numeric_df.corr(), text_auto=True, aspect='auto', color_continuous_scale='RdBu_r')
        fig.update_layout(title="Korrelationsmatrix")
        fig.show()

interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …

In [7]:
@interact(df_name=['df1', 'df2', 'df3'])
def correlate_with_target(df_name):
    df = get_df(df_name)
    num_cols = df.select_dtypes(include='number').columns.tolist()

    @interact(target_variable=num_cols)
    def inner(target_variable):
        if target_variable not in df.columns:
            print("Zielvariable nicht vorhanden.")
            return

        corr_series = df.corr(numeric_only=True)[target_variable].drop(target_variable).sort_values()

        if corr_series.empty:
            print("Keine korrelierenden numerischen Merkmale gefunden.")
            return

        corr_df = corr_series.to_frame(name='Korrelationskoeffizient').reset_index()
        corr_df.rename(columns={'index': 'Feature'}, inplace=True)

        fig = px.bar(
            corr_df,
            x='Korrelationskoeffizient',
            y='Feature',
            orientation='h',
            color='Feature',
            color_discrete_sequence=px.colors.qualitative.Set3,
            text='Korrelationskoeffizient',
            title=f"Korrelation mit Zielvariable: {target_variable}"
        )
        fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
        fig.update_layout(yaxis={'categoryorder': 'total ascending'})
        fig.show()


interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …

In [8]:

@interact(df_name=['df1', 'df2', 'df3'])
def plot_feature_importance(df_name):
    df = get_df(df_name).dropna()
    all_cols = df.columns.tolist()

    @interact(target_variable=all_cols)
    def inner(target_variable):
        if target_variable not in df.columns:
            print("Zielvariable nicht vorhanden.")
            return

        # Alle numerischen Spalten außer Ziel als Prädiktoren
        X = df.select_dtypes(include='number').drop(columns=[target_variable], errors='ignore')
        y = df[target_variable]

        # Modell nur trainieren, wenn genügend Varianz & Features vorhanden
        if y.nunique() > 1 and len(X.columns) > 0:
            if not pd.api.types.is_numeric_dtype(y):
                y = LabelEncoder().fit_transform(y)

            model = RandomForestClassifier(n_estimators=100, random_state=0)
            model.fit(X, y)

            # Feature Importances
            importances = pd.Series(model.feature_importances_, index=X.columns)
            imp_df = importances.sort_values(ascending=True).tail(20).reset_index()
            imp_df.columns = ["Feature", "Importance"]

            # Plot
            fig = px.bar(
                imp_df,
                x="Importance",
                y="Feature",
                orientation="h",
                color="Feature",  # jede Feature bekommt eine eigene Farbe
                color_discrete_sequence=px.colors.qualitative.Set3,  # gut unterscheidbare Palette
                text="Importance",
                title=f"Top Merkmale für Vorhersage von '{target_variable}'",
                labels={"Importance": "Feature Importance", "Feature": "Feature"}
            )

            fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
            fig.update_layout(
                yaxis_title="Feature",
                xaxis_title="Bedeutung",
                legend_title="Feature",
                bargap=0.2
            )
            fig.show()

        else:
            print("Nicht geeignet für Modelltraining (zu wenig Varianz oder Features).")


interactive(children=(Dropdown(description='df_name', options=('df1', 'df2', 'df3'), value='df1'), Output()), …